# Thesis Methodology Figures

This notebook generates **methodology-only** figures for the thesis.

It intentionally avoids result figures and focuses on the methodological story:

1. dataset selection
2. story coverage
3. transcript timing and TR alignment
4. model layer sampling
5. brain target spaces
6. shared predictor design for brain and LM fitting

When executed, the notebook saves all figures into:

`thesis_methodology/figures/methodology`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "eda_brain_data").exists() and (candidate / "structure_comparison").exists():
            return candidate
    raise RuntimeError("Could not locate repo root.")


ROOT = find_repo_root()
FIG_DIR = ROOT / "thesis_methodology" / "figures" / "methodology"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
    }
)


def save_current_figure(name: str) -> Path:
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches="tight")
    return path


TRANSCRIPT_ROOT = ROOT / "eda_brain_data" / "datasets" / "data" / "transcripts"
ROI_ROOT = ROOT / "eda_brain_data" / "assets" / "roi"
DS002345_ROOT = ROOT / "eda_brain_data" / "datasets" / "ds002345"

DATASET_COMPARISON = pd.DataFrame(
    [
        {
            "dataset": "ds002345 Narratives",
            "subjects": 105,
            "runs": 211,
            "stories": 6,
            "role": "main dataset",
        },
        {
            "dataset": "ds002322 Alice",
            "subjects": 26,
            "runs": 26,
            "stories": 1,
            "role": "reference dataset",
        },
    ]
)

TARGET_STORIES = ["black", "bronx", "forgot", "piemanpni", "shapesphysical", "shapessocial"]

story_rows = []
for story in TARGET_STORIES:
    count = 0
    for path in DS002345_ROOT.glob(f"sub-*/func/*_task-{story}_events.tsv"):
        count += 1
    story_rows.append({"story": story, "runs": count, "subjects": count})
story_df = pd.DataFrame(story_rows)

shape_meta = {
    slug: json.loads((TRANSCRIPT_ROOT / slug / "metadata.json").read_text(encoding="utf-8"))
    for slug in ["shapesphysical", "shapessocial"]
}

shapes_words = pd.read_csv(
    TRANSCRIPT_ROOT / "shapesphysical" / "shapesphysical_words.tsv",
    sep="\t",
)
shapes_tr = pd.read_csv(
    TRANSCRIPT_ROOT / "shapesphysical" / "shapesphysical_tr_aligned.tsv",
    sep="\t",
)

MODEL_LAYERS = {
    "Gemma 2 2B": {
        "model_id": "google/gemma-2-2b",
        "scope_release": "gemma-scope-2b-pt-res-canonical",
        "scope_width_label": "width_16k",
        "latent_width": 16384,
        "d_model": 2304,
        "intermediate_size": 9216,
        "total_layers": 26,
        "selected_layers": [4, 8, 13, 17, 22, 25],
        "num_attention_heads": 8,
        "num_key_value_heads": 4,
        "head_dim": 256,
        "max_position_embeddings": 8192,
        "vocab_size": 256000,
        "token_layer": 4,
    },
    "Gemma 2 9B": {
        "model_id": "google/gemma-2-9b",
        "scope_release": "gemma-scope-9b-pt-res-canonical",
        "scope_width_label": "width_16k",
        "latent_width": 16384,
        "d_model": 3584,
        "intermediate_size": 14336,
        "total_layers": 42,
        "selected_layers": [7, 13, 21, 27, 36, 40],
        "num_attention_heads": 16,
        "num_key_value_heads": 8,
        "head_dim": 256,
        "max_position_embeddings": 8192,
        "vocab_size": 256000,
        "token_layer": 7,
    },
    "Llama 3.1 8B": {
        "model_id": "meta-llama/Llama-3.1-8B",
        "scope_release": "llama_scope_lxr_8x",
        "scope_width_label": "8x",
        "latent_width": 32768,
        "d_model": 4096,
        "intermediate_size": 14336,
        "total_layers": 32,
        "selected_layers": [5, 10, 16, 21, 27, 31],
        "num_attention_heads": 32,
        "num_key_value_heads": 8,
        "head_dim": 128,
        "max_position_embeddings": 131072,
        "vocab_size": 128256,
        "token_layer": 5,
    },
}

MODEL_STRUCTURE_DF = pd.DataFrame(
    [
        {
            "model": model_name,
            "model_id": spec["model_id"],
            "sae_release": spec["scope_release"],
            "sae_width": spec["scope_width_label"],
            "latent_width": spec["latent_width"],
            "d_model": spec["d_model"],
            "n_over_d": round(spec["latent_width"] / spec["d_model"], 2),
            "intermediate_size": spec["intermediate_size"],
            "layers": spec["total_layers"],
            "selected_layers": ",".join(str(x) for x in spec["selected_layers"]),
            "attention_heads": spec["num_attention_heads"],
            "kv_heads": spec["num_key_value_heads"],
            "head_dim": spec["head_dim"],
            "context_window": spec["max_position_embeddings"],
            "vocab_size": spec["vocab_size"],
        }
        for model_name, spec in MODEL_LAYERS.items()
    ]
)

display(Markdown(f"Figures will be saved to: `{FIG_DIR}`"))
display(DATASET_COMPARISON)
display(story_df)
display(MODEL_STRUCTURE_DF)


## Figure 1. Methodology Pipeline Overview

This figure gives a compact end-to-end overview of the retained thesis methodology:

- selective dataset curation
- transcript timing and alignment
- transcript-first SAE feature extraction
- TR aggregation
- cleaned brain targets
- matched brain and LM regressions
- structural comparison metrics


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

pipeline_boxes = [
    (0.03, 0.58, 0.14, 0.22, "OpenNeuro\nselection\n(ds002345, ds002322)", "#dbeafe"),
    (0.20, 0.58, 0.15, 0.22, "Selective retrieval\nand EDA\n(metadata first)", "#e0f2fe"),
    (0.38, 0.58, 0.15, 0.22, "Transcript timing\nWhisper + Gentle\nword/TR assets", "#dcfce7"),
    (0.56, 0.58, 0.15, 0.22, "Transcript-first SAE\nfeature discovery\nand ranking", "#fef3c7"),
    (0.74, 0.58, 0.12, 0.22, "TR aggregation\naverage view", "#fde68a"),
    (0.25, 0.18, 0.18, 0.22, "fMRIPrep + cleaning\nconfounds, censoring,\nrun-wise z-score", "#fee2e2"),
    (0.48, 0.18, 0.18, 0.22, "Brain targets\nSchaefer-200 +\nROI sensitivity", "#fecaca"),
    (0.71, 0.18, 0.22, 0.22, "Matched regressions\nbrain and final hidden state\nthen compare r, R2, RSA", "#ede9fe"),
]

for x, y, w, h, text, color in pipeline_boxes:
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.015,rounding_size=0.03",
        linewidth=1.5,
        edgecolor="#334155",
        facecolor=color,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

arrow_specs = [
    ((0.17, 0.69), (0.20, 0.69)),
    ((0.35, 0.69), (0.38, 0.69)),
    ((0.53, 0.69), (0.56, 0.69)),
    ((0.71, 0.69), (0.74, 0.69)),
    ((0.46, 0.58), (0.34, 0.40)),
    ((0.81, 0.58), (0.81, 0.40)),
    ((0.43, 0.29), (0.48, 0.29)),
    ((0.66, 0.29), (0.71, 0.29)),
]

for start, end in arrow_specs:
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color="#475569",
        )
    )

ax.set_title("Thesis Methodology Overview", pad=18)
saved = save_current_figure("01_methodology_pipeline_overview.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 2. Dataset Comparison

This figure justifies why the thesis centered on Narratives while keeping Alice as a reference dataset.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
metrics = [("subjects", "Subjects"), ("runs", "BOLD runs"), ("stories", "Story conditions")]
colors = ["#2563eb", "#0ea5e9"]

for ax, (column, title) in zip(axes, metrics):
    bars = ax.bar(DATASET_COMPARISON["dataset"], DATASET_COMPARISON[column], color=colors, width=0.65)
    ax.set_title(title)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)
    for bar, value in zip(bars, DATASET_COMPARISON[column]):
        ax.text(bar.get_x() + bar.get_width() / 2, value, str(value), ha="center", va="bottom")

fig.suptitle("Candidate Dataset Comparison", y=1.03, fontsize=15)
plt.tight_layout()
saved = save_current_figure("02_dataset_comparison.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 3. Narratives Story Coverage

This figure shows the six retained Narratives stories used in the expanded stimulus base.


In [ ]:
story_plot_df = story_df.copy()
story_plot_df["story_type"] = [
    "additional",
    "additional",
    "additional",
    "additional",
    "primary",
    "matched companion",
]

color_map = {
    "primary": "#dc2626",
    "matched companion": "#f59e0b",
    "additional": "#2563eb",
}

fig, ax = plt.subplots(figsize=(10.5, 4.8))
bars = ax.bar(
    story_plot_df["story"],
    story_plot_df["runs"],
    color=[color_map[v] for v in story_plot_df["story_type"]],
    width=0.68,
)
ax.set_title("Narratives Story Coverage Used For Collection And Expansion")
ax.set_ylabel("Subject-runs")
ax.set_xlabel("Story")
ax.tick_params(axis="x", rotation=20)

for bar, row in zip(bars, story_plot_df.itertuples(index=False)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        row.runs,
        f"{row.runs}",
        ha="center",
        va="bottom",
    )

legend_handles = [
    Rectangle((0, 0), 1, 1, color=color_map[key], label=key)
    for key in ["primary", "matched companion", "additional"]
]
ax.legend(handles=legend_handles, title="Role", frameon=False)
plt.tight_layout()
saved = save_current_figure("03_narratives_story_coverage.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 4. Shapes Timing Structure

This figure shows why the Shapes stories were especially convenient methodologically: matched TR, matched onset correction, and nearly identical total timing.


In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 3.8))
y_positions = {"shapesphysical": 1, "shapessocial": 0}
colors = {"music": "#93c5fd", "story": "#86efac"}

for slug, meta in shape_meta.items():
    y = y_positions[slug]
    onset = float(meta["stimulus_onset_s"])
    end = float(meta["stimulus_end_s"])
    ax.barh(y, onset, left=0, color=colors["music"], edgecolor="none", height=0.45)
    ax.barh(y, end - onset, left=onset, color=colors["story"], edgecolor="none", height=0.45)
    ax.text(onset / 2, y, "intro/music", ha="center", va="center", fontsize=10)
    ax.text(onset + (end - onset) / 2, y, "narrative segment", ha="center", va="center", fontsize=10)
    ax.text(end + 4, y, f"{end:.1f}s", va="center", fontsize=10)

ax.axvline(4.5, color="#1f2937", linestyle="--", linewidth=1.5, label="stimulus onset correction")
ax.set_yticks([1, 0])
ax.set_yticklabels(["shapesphysical", "shapessocial"])
ax.set_xlabel("Time (s)")
ax.set_title("Matched Timing Structure For The Shapes Stories")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
saved = save_current_figure("04_shapes_timing_structure.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 5. Word-To-TR Alignment Example

This figure uses the real `shapesphysical` transcript assets to show how word timings are shifted onto the TR grid.


In [ ]:
meta = shape_meta["shapesphysical"]
onset = float(meta["stimulus_onset_s"])
window_start = onset
window_end = onset + 12.0
word_subset = shapes_words.copy()
word_subset["aligned_start_s"] = word_subset["start_s"] + onset
word_subset["aligned_end_s"] = word_subset["end_s"] + onset
word_subset = word_subset[(word_subset["aligned_end_s"] >= window_start) & (word_subset["aligned_start_s"] <= window_end)].copy()
tr_subset = shapes_tr[(shapes_tr["start_s"] >= 0) & (shapes_tr["start_s"] <= 12.0)].copy()
tr_subset["aligned_start_s"] = tr_subset["start_s"] + onset
tr_subset["aligned_end_s"] = tr_subset["end_s"] + onset

fig, ax = plt.subplots(figsize=(13, 4.2))
for row in tr_subset.itertuples(index=False):
    ax.axvspan(row.aligned_start_s, row.aligned_end_s, color="#e2e8f0", alpha=0.6)
    ax.text(
        (row.aligned_start_s + row.aligned_end_s) / 2,
        1.28,
        f"TR {int(row.tr_index)}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#334155",
    )

for idx, row in enumerate(word_subset.itertuples(index=False)):
    y = 0.75 - 0.18 * (idx % 2)
    ax.add_patch(
        Rectangle(
            (row.aligned_start_s, y),
            row.aligned_end_s - row.aligned_start_s,
            0.12,
            facecolor="#60a5fa",
            edgecolor="#1d4ed8",
            alpha=0.85,
        )
    )
    ax.text(
        (row.aligned_start_s + row.aligned_end_s) / 2,
        y + 0.06,
        row.word,
        ha="center",
        va="center",
        fontsize=9,
        color="white",
    )

ax.axvline(onset, color="#dc2626", linestyle="--", linewidth=1.6)
ax.text(onset, 1.43, "story onset", ha="center", va="bottom", color="#dc2626")
ax.set_xlim(window_start, window_end)
ax.set_ylim(0.35, 1.48)
ax.set_xlabel("Scan time (s)")
ax.set_yticks([])
ax.set_title("Example Of Timed Words Assigned To The fMRI TR Grid")
plt.tight_layout()
saved = save_current_figure("05_word_to_tr_alignment_example.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 6. Relative-Depth Layer Sampling

This figure shows how the selected analysis layers cover comparable early, middle, and late depth positions across the three architectures.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
model_names = list(MODEL_LAYERS.keys())
x_positions = np.arange(len(model_names))

for x, model_name in enumerate(model_names):
    total = MODEL_LAYERS[model_name]["total_layers"]
    selected = MODEL_LAYERS[model_name]["selected_layers"]
    ax.plot([x, x], [1, total], color="#cbd5e1", linewidth=6, solid_capstyle="round", zorder=1)
    ax.scatter([x] * len(selected), selected, s=110, color="#2563eb", zorder=3)
    for layer in selected:
        ax.text(x + 0.06, layer, str(layer), va="center", fontsize=9)

ax.set_xticks(x_positions)
ax.set_xticklabels(model_names)
ax.set_ylabel("Layer index")
ax.set_title("Selected Layers Sample Comparable Relative Depths Across Models")
ax.invert_yaxis()
plt.tight_layout()
saved = save_current_figure("06_relative_depth_layer_sampling.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Model Architecture Summary

The next cells are intentionally more detailed than the shared layer-sampling figure. Each model gets its own panel showing:

- transformer depth
- which residual layers had matching SAEs
- which layers were selected for the thesis
- the residual width `D`
- the SAE latent width `N`
- the ratio `N / D`
- attention and MLP dimensions


In [ ]:
display(Markdown("## Model / SAE Structure Table"))
display(MODEL_STRUCTURE_DF)


In [ ]:
def plot_model_structure_panel(model_name: str, spec: dict, figure_name: str) -> Path:
    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.2])
    ax_layers = fig.add_subplot(gs[0, 0])
    ax_specs = fig.add_subplot(gs[0, 1])

    total_layers = int(spec["total_layers"])
    selected_layers = list(spec["selected_layers"])
    token_layer = int(spec["token_layer"])

    ax_layers.plot([0, 0], [1, total_layers], color="#cbd5e1", linewidth=12, solid_capstyle="round", zorder=1)
    all_layers = np.arange(1, total_layers + 1)
    ax_layers.scatter(np.zeros_like(all_layers), all_layers, s=18, color="#94a3b8", zorder=2)
    ax_layers.scatter(np.zeros(len(selected_layers)), selected_layers, s=140, color="#2563eb", zorder=3)
    ax_layers.scatter([0], [token_layer], s=180, color="#dc2626", marker="D", zorder=4)

    for layer in selected_layers:
        fraction = layer / total_layers
        ax_layers.text(0.08, layer, f"layer {layer} ({fraction:.0%})", va="center", fontsize=10)

    if token_layer not in selected_layers:
        ax_layers.text(0.08, token_layer, f"token stream layer {token_layer}", va="center", fontsize=10, color="#dc2626")
    else:
        ax_layers.text(0.08, token_layer - 0.9, f"canonical token stream", va="center", fontsize=9, color="#dc2626")

    ax_layers.set_xlim(-0.2, 1.05)
    ax_layers.set_ylim(total_layers + 1, 0)
    ax_layers.set_xticks([])
    ax_layers.set_ylabel("Transformer layer index")
    ax_layers.set_title(f"{model_name}: transformer depth and selected SAE layers")

    ax_specs.axis("off")
    info_lines = [
        f"Base model: {spec['model_id']}",
        f"SAE release: {spec['scope_release']}",
        f"SAE width label: {spec['scope_width_label']}",
        f"Residual width D: {spec['d_model']:,}",
        f"SAE latent width N: {spec['latent_width']:,}",
        f"N / D ratio: {spec['latent_width'] / spec['d_model']:.2f}x",
        f"MLP / intermediate size: {spec['intermediate_size']:,}",
        f"Attention heads: {spec['num_attention_heads']}",
        f"KV heads: {spec['num_key_value_heads']}",
        f"Head dim: {spec['head_dim']}",
        f"Context window: {spec['max_position_embeddings']:,}",
        f"Vocabulary size: {spec['vocab_size']:,}",
        f"Selected layers: {', '.join(str(x) for x in selected_layers)}",
    ]

    box = FancyBboxPatch(
        (0.02, 0.05),
        0.94,
        0.88,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        linewidth=1.4,
        edgecolor="#334155",
        facecolor="#f8fafc",
    )
    ax_specs.add_patch(box)
    ax_specs.text(0.05, 0.92, f"{model_name}: retained architecture summary", fontsize=13, weight="bold", va="top")
    ax_specs.text(0.05, 0.86, "\n".join(info_lines), fontsize=11, va="top", linespacing=1.45)

    fig.suptitle(f"{model_name} SAE / Transformer Structure", y=0.98, fontsize=16)
    plt.tight_layout()
    saved_path = save_current_figure(figure_name)
    plt.show()
    return saved_path


for model_name, spec, figure_name in [
    ("Gemma 2 2B", MODEL_LAYERS["Gemma 2 2B"], "09_gemma_2_2b_sae_structure.png"),
    ("Gemma 2 9B", MODEL_LAYERS["Gemma 2 9B"], "10_gemma_2_9b_sae_structure.png"),
    ("Llama 3.1 8B", MODEL_LAYERS["Llama 3.1 8B"], "11_llama_3_1_8b_sae_structure.png"),
]:
    saved = plot_model_structure_panel(model_name, spec, figure_name)
    display(Markdown(f"Saved: `{saved}`"))


In [ ]:
comparison_df = MODEL_STRUCTURE_DF.copy()
plot_cols = ["d_model", "latent_width", "intermediate_size"]
labels = {
    "d_model": "Residual width D",
    "latent_width": "SAE latent width N",
    "intermediate_size": "MLP width",
}

fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(comparison_df))
width = 0.22
colors = ["#2563eb", "#dc2626", "#16a34a"]

for idx, col in enumerate(plot_cols):
    bars = ax.bar(x + (idx - 1) * width, comparison_df[col], width=width, label=labels[col], color=colors[idx])
    for bar, value in zip(bars, comparison_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{int(value):,}", ha="center", va="bottom", fontsize=9, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(comparison_df["model"])
ax.set_ylabel("Width / dimension")
ax.set_title("Cross-Model Width Breakdown For The Retained SAE / Transformer Setup")
ax.legend(frameon=False)
plt.tight_layout()
saved = save_current_figure("12_model_width_breakdown.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 7. Brain Target Spaces

This figure contrasts the main cortical target space with the secondary non-cortical ROI sensitivity analysis.


In [ ]:
def load_mid_slices(path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    data = nib.load(str(path)).get_fdata()
    mids = [dim // 2 for dim in data.shape[:3]]
    axial = np.rot90(data[:, :, mids[2]])
    coronal = np.rot90(data[:, mids[1], :])
    sagittal = np.rot90(data[mids[0], :, :])
    return axial, coronal, sagittal


schaefer_path = ROI_ROOT / "Schaefer2018_200Parcels_7Networks_order_FSLMNI152_1mm.nii.gz"
roi_path = ROI_ROOT / "subcortical_cerebellar_spheres_1mm.nii.gz"

schaefer_slices = load_mid_slices(schaefer_path)
roi_slices = load_mid_slices(roi_path)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, view_name in enumerate(["Axial", "Coronal", "Sagittal"]):
    axes[0, col].imshow(schaefer_slices[col], cmap="tab20", interpolation="nearest")
    axes[0, col].set_title(f"Schaefer-200 {view_name}")
    axes[0, col].axis("off")
    axes[1, col].imshow(roi_slices[col], cmap="tab10", interpolation="nearest")
    axes[1, col].set_title(f"ROI sensitivity {view_name}")
    axes[1, col].axis("off")

fig.suptitle("Main Cortical Target Space And Secondary ROI Sensitivity Space", y=0.98, fontsize=15)
plt.tight_layout()
saved = save_current_figure("07_brain_target_spaces.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure 8. Shared Predictor Design

This diagram summarizes the final retained fitting design:

- shared transcript-grounded predictor basis
- brain branch with fixed FIR lags
- LM branch with same-TR final hidden-state targets
- structural comparison after fitting


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

boxes = [
    (0.05, 0.35, 0.18, 0.28, "Transcript-grounded\npredictor basis\n(top-k = 128, average TR)", "#fef3c7"),
    (0.34, 0.62, 0.22, 0.20, "Brain design matrix\nlagged TR copies\n(0..4)", "#fee2e2"),
    (0.34, 0.18, 0.22, 0.20, "LM design matrix\nsame-TR predictors\n(main method)", "#dbeafe"),
    (0.66, 0.62, 0.18, 0.20, "Cleaned brain targets\nSchaefer parcels", "#fecaca"),
    (0.66, 0.18, 0.18, 0.20, "Final hidden-state\ntargets", "#bfdbfe"),
    (0.86, 0.38, 0.10, 0.24, "Compare\nR, R2, RSA,\nfeature overlap", "#e9d5ff"),
]

for x, y, w, h, text, color in boxes:
    ax.add_patch(
        FancyBboxPatch(
            (x, y),
            w,
            h,
            boxstyle="round,pad=0.015,rounding_size=0.03",
            linewidth=1.5,
            edgecolor="#334155",
            facecolor=color,
        )
    )
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

arrows = [
    ((0.23, 0.53), (0.34, 0.72)),
    ((0.23, 0.45), (0.34, 0.28)),
    ((0.56, 0.72), (0.66, 0.72)),
    ((0.56, 0.28), (0.66, 0.28)),
    ((0.84, 0.72), (0.86, 0.53)),
    ((0.84, 0.28), (0.86, 0.47)),
]
for start, end in arrows:
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color="#475569",
        )
    )

ax.text(0.45, 0.88, "brain branch", ha="center", va="center", fontsize=11, color="#991b1b")
ax.text(0.45, 0.08, "LM branch", ha="center", va="center", fontsize=11, color="#1d4ed8")
ax.set_title("Final Retained Shared-Predictor Design", pad=18)
saved = save_current_figure("08_shared_predictor_design.png")
plt.show()
display(Markdown(f"Saved: `{saved}`"))


## Figure Inventory

The cell below gives a quick inventory of all saved methodology figures.


In [ ]:
inventory_rows = []
for path in sorted(FIG_DIR.glob("*.png")):
    inventory_rows.append({"file": path.name, "size_kb": round(path.stat().st_size / 1024, 1)})

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)
display(Markdown(f"Total figures saved: **{len(inventory_df)}**"))
